In [1]:
!pip install xgboost

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import median_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
from scipy import stats

In [2]:
mlb_pitchers = pd.read_csv('stored_datasets/All Pitchers Cleaned.csv')
mlb_pitchers

,Player,Debut Year,Debut Age,Retirement Year,Retirement Age,Career Length,Wins,Losses,Win Percentage,Total Decisions,...,BF,ERA+,FIP,WHIP,H9,HR9,BB9,SO9,SO/BB,Hall of Fame
0,Cy Young,1890,23,1911,44,21,511,315,0.619,826,...,29565,138,2.84,1.130,8.7,0.2,1.5,3.4,2.30,1
1,Pud Galvin,1875,18,1892,35,17,365,310,0.541,675,...,25415,107,2.96,1.191,9.6,0.2,1.1,2.7,2.43,1
2,Walter Johnson,1907,19,1927,39,20,417,279,0.599,696,...,23415,147,2.38,1.061,7.5,0.1,2.1,5.3,2.57,1
3,Phil Niekro,1964,25,1987,48,23,318,274,0.537,592,...,22677,115,3.62,1.268,8.4,0.8,3.0,5.6,1.85,1
4,Nolan Ryan,1966,19,1993,46,27,324,292,0.526,616,...,22575,112,2.97,1.247,6.6,0.5,4.7,9.5,2.04,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1210,Charlie Robertson,1919,23,1928,32,9,49,80,0.380,129,...,4453,90,3.89,1.518,10.3,0.3,3.4,2.8,0.82,0
1211,Sammy Ellis,1962,21,1969,28,7,63,58,0.521,121,...,4296,88,3.90,1.340,8.7,1.1,3.4,6.1,1.79,0
1212,Lil Stoner,1922,23,1931,32,9,50,57,0.467,107,...,4466,87,4.13,1.548,10.6,0.6,3.4,2.7,0.80,0
1213,Johnny Humphries,1938,23,1946,31,8,52,63,0.452,115,...,4342,97,3.80,1.394,9.2,0.4,3.4,2.8,0.85,0


In [3]:
xgb_model = xgb.XGBClassifier()

# Features
X = mlb_pitchers.drop(columns=['Hall of Fame', 'Player'])

# Target
y = mlb_pitchers['Hall of Fame']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [16]:
%%time

# Define two types of K-Fold Cross Validation strategies so that either one can be used
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Define model
xgb_model = xgb.XGBClassifier()

param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 4, 5, 6],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 1, 5],
    "reg_alpha": [0, 0.1, 1],   # L1 regularization
    "reg_lambda": [0, 0.1, 1]   # L2 regularization
}

# Define GridSearchCV with KFold
grid_search = GridSearchCV(
    estimator = xgb_model,
    param_grid=param_grid,
    cv=kf,
    scoring='precision',
    n_jobs=-1
)

# Fit on your training data
grid_search.fit(X_train, y_train)

# Best model
#best_model = grid_search.best_estimator_
xgb_model = grid_search.best_estimator_
xgb_model.fit(X_train, y_train)

CPU times: user 57.8 s, sys: 18.3 s, total: 1min 16s
Wall time: 21min 59s


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.6, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=5, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [17]:
import joblib
#joblib.dump(xgb_model, 'stored_models/model_xgb_best_recall_stratified_kfold.sav')
#joblib.dump(xgb_model, 'stored_models/model_xgb_best_recall_standard_kfold.sav')
#joblib.dump(xgb_model, 'stored_models/model_xgb_best_F1_standard_kfold.sav')
#joblib.dump(xgb_model, 'stored_models/model_xgb_best_F1_stratified_kfold.sav')
#joblib.dump(xgb_model, 'stored_models/model_xgb_best_precision_stratified_kfold.sav')
joblib.dump(xgb_model, 'stored_models/model_xgb_best_precision_standard_kfold.sav')

['stored_models/model_xgb_best_precision_standard_kfold.sav']

In [18]:
# Define the same KFold strategy
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Get out-of-fold predictions using the tuned model
y_pred_all_best = cross_val_predict(xgb_model, X, y, cv=kf)

# Get predicted probabilities
y_proba_all_best = cross_val_predict(xgb_model, X, y, cv=kf, method='predict_proba')

# Regular predictions
y_pred_all_best = cross_val_predict(xgb_model, X, y, cv=kf)

# If binary classification, pick the probability of the predicted class
# (confidence in whichever class was chosen)
y_pred_indices = np.array([list(np.unique(y)).index(p) for p in y_pred_all_best])
confidence_scores = y_proba_all_best[np.arange(len(y_proba_all_best)), y_pred_indices]

# Build results dataframe
xgb_gs_results = pd.DataFrame()
xgb_gs_results['Player'] = mlb_pitchers['Player']
xgb_gs_results['Hall of Fame'] = y
xgb_gs_results['Prediction'] = y_pred_all_best
xgb_gs_results['Prediction Correct'] = (xgb_gs_results['Hall of Fame'] == xgb_gs_results['Prediction']).astype(int)
xgb_gs_results['Prediction Confidence'] = confidence_scores
xgb_gs_results

,Player,Hall of Fame,Prediction,Prediction Correct,Prediction Confidence
0,Cy Young,1,1,1,0.594792
1,Pud Galvin,1,0,0,0.691684
2,Walter Johnson,1,1,1,0.591834
3,Phil Niekro,1,1,1,0.559220
4,Nolan Ryan,1,0,0,0.587846
...,...,...,...,...,...
1210,Charlie Robertson,0,0,1,0.940315
1211,Sammy Ellis,0,0,1,0.941558
1212,Lil Stoner,0,0,1,0.941031
1213,Johnny Humphries,0,0,1,0.940315


In [19]:
#xgb_gs_results.to_csv('stored_results/Results_XGB_GS_Best_Recall_Stratified_KFold.csv', index=False)
#xgb_gs_results.to_csv('stored_results/Results_XGB_GS_Best_Recall_Standard_KFold.csv', index=False)
#xgb_gs_results.to_csv('stored_results/Results_XGB_GS_Best_F1_Standard_KFold.csv', index=False)
#xgb_gs_results.to_csv('stored_results/Results_XGB_GS_Best_F1_Stratified_KFold.csv', index=False)
#xgb_gs_results.to_csv('stored_results/Results_XGB_GS_Best_Precision_Stratified_KFold.csv', index=False)
#xgb_gs_results.to_csv('stored_results/Results_XGB_GS_Best_Precision_Standard_KFold.csv', index=False)

In [ ]:
#xbg_gs_results.to_csv('stored_results/Results_XGB_GS_Best_Precision_Standard_KFold.csv', index=False)


#joblib.dump(xgb_model, 'stored_models/model_xgb_best_recall_stratified_kfold.sav')

In [ ]:
#gbc_gs_results.to_csv('stored_results/Results_GBC_GS_Best_Precision_Standard_KFold.csv', index=False)


# No GridSearch